# behaviors

> The client runtime: delegated event rules that survive htmx swaps

In [ ]:
#| default_exp behaviors

The registry stores a component's behaviors as records: a selector, an event, a handler body. Here we turn those into running code in the browser. Wiring a listener onto each element as it appears would work for a static page, but in a FastHTML app elements arrive continuously as htmx swaps in fragments, so per-element wiring would mean re-initialising after every swap.

Instead we delegate: every rule becomes a document-level listener which, when its event fires anywhere on the page, checks whether the target sits inside an element matching the rule's selector. Matching happens at event time against the live DOM, so elements swapped in later behave the same as those in the original page, with no per-element state and no mutation observer. The rule set is plain data, emitted fresh on each page render, which also lets a hyperelement be redefined in a live notebook.

In [ ]:
import json
from fastcore.utils import *
from answerui.registry import _registry

In [ ]:
from answerui.registry import Rules, hypelt

## The runtime

The script below is the whole runtime. Rules arrive as a list of records, each with a selector, an event name, a capture flag, and a handler function. We group them by event and phase, so each pair gets one listener however many rules share it. On an event, each rule in the group looks for the nearest ancestor of the target (the target included) that matches its selector, and calls the handler with it.

`htmx:load` fires on the root of newly swapped content, and what we want to match is that content: the root itself and anything inside it. A `deep` rule therefore searches downward from the target instead of upward. Walking up would run a handler for an ancestor that was already on the page, as though it had just arrived.

Handlers are async functions, so a failure in one rejects its own promise and leaves the other rules on that event running. Listening in the capture phase is opt-in, for events like `toggle` and `scroll` that don't bubble.

In [ ]:
_runtime = r"""(() => {
const rules = [%s];
const by = {};
rules.forEach(r => (by[r.ev + '|' + (r.cap ? 1 : 0)] ??= []).push(r));
for (const k in by) {
  const [ev, cap] = k.split('|');
  document.addEventListener(ev, e => {
    const t = e.target;
    if (!(t instanceof Element)) return;
    for (const r of by[k]) {
      if (r.deep) {
        if (t.matches(r.sel)) r.fn(t, e);
        t.querySelectorAll(r.sel).forEach(el => r.fn(el, e));
      } else {
        const el = t.closest(r.sel);
        if (el) r.fn(el, e);
      }
    }
  }, {capture: cap === '1'});
}
})();"""

## Entries

Each registry record becomes one object literal in that list. Event names pass through as written, with one convention: a leading `:` means the htmx event, so `':after-request'` listens for `htmx:after-request`. The `:` form is only a shorthand, so `deep` is set for `htmx:load` whichever way it is written. The handler body is inserted verbatim into an async function taking `el` and `e`:

In [ ]:
def _entry(r):
    ev = r['ev']
    if ev.startswith(':'): ev = 'htmx'+ev
    deep = ev=='htmx:load'
    return (f'{{sel: {json.dumps(r["sel"])}, ev: {json.dumps(ev)}, cap: {json.dumps(r["cap"])}, '
        f'deep: {json.dumps(deep)}, async fn(el, e) {{ {r["js"]} }}}}')

`build_js` collects the behaviors of every registry entry, in registration order, and fills the runtime.

In [ ]:
def build_js():
    "The behavior script: every registered entry's behaviors, in registration order"
    return _runtime % ',\n'.join(_entry(r) for o in _registry.values() for r in o.on)

As examples, let's make a notice that removes itself after a delay (or when a child marked `dismiss` is clicked), and a menu that focuses its first item when its popover opens. The autodismiss listens for `:load`, which htmx fires for the initial page and for every swapped fragment, but never for content already on the page, so swapping something inside a notice doesn't restart it. Since `toggle` doesn't bubble, the menu rule uses the capture phase:

In [ ]:
NoticeBox = hypelt('NoticeBox', layer='components', doc='A notice that dismisses itself.', on={
    ('&[autodismiss]', ':load'): r"await new Promise(r => setTimeout(r, +el.getAttribute('autodismiss'))); el.remove()",
    ('& [dismiss]', 'click'): r"el.closest('notice-box').remove()"})
Rules('menu', sel='[popover][menu]', on={('&', 'toggle', True): r"if (e.newState === 'open') el.querySelector('a, button')?.focus()"})
print('\n'.join(l for l in build_js().splitlines() if 'notice-box' in l or '[menu]' in l))

{sel: "notice-box[autodismiss]", ev: "htmx:load", cap: false, deep: true, async fn(el, e) { await new Promise(r => setTimeout(r, +el.getAttribute('autodismiss'))); el.remove() }},
{sel: "notice-box [dismiss]", ev: "click", cap: false, deep: false, async fn(el, e) { el.closest('notice-box').remove() }},
{sel: "[popover][menu]", ev: "toggle", cap: true, deep: false, async fn(el, e) { if (e.newState === 'open') el.querySelector('a, button')?.focus() }}];


The `:load` rule became `htmx:load` with `deep` set, the `toggle` rule carries `cap`, and the click rule is plain. The browser does the grouping into listeners, so the script itself is only the rule list.

In [ ]:
s = build_js()
assert '"htmx:load"' in s and 'deep: true' in s and 'cap: true' in s

## Export

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()